# TuneForge — Experiment 001

**Goal:** run the canonical TinyLlama + Dolly LoRA experiment on a real NVIDIA T4 and generate auditable launch evidence.

This notebook uses the merged TuneForge engine. It does **not** recreate the original Scaler notebook workflow.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before continuing.')
gpu = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print('GPU:', gpu)
print('Compute capability:', f'{capability[0]}.{capability[1]}')
if 'T4' not in gpu:
    raise RuntimeError(f'Experiment 001 expects a T4 runtime; detected {gpu}.')


In [ ]:
!rm -rf /content/TuneForge
!git clone https://github.com/mohit231007/TuneForge.git /content/TuneForge
%cd /content/TuneForge
!git rev-parse HEAD


In [ ]:
!python -m pip install --upgrade pip -q
!python -m pip install -U 'torchao>=0.16.0' -q
!python -m pip install -e . -q


In [ ]:
import torch, transformers, peft, trl
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', '.'.join(map(str, torch.cuda.get_device_capability(0))))
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)


In [ ]:
!python -m tuneforge.experiment_001 --config configs/experiment_001.yaml --clean


In [ ]:
import json
from pathlib import Path
evidence_path = Path('artifacts/experiment-001/experiment_001_evidence.json')
evidence = json.loads(evidence_path.read_text())
summary = {
    'git_commit': evidence['provenance']['git_commit'],
    'gpu': evidence['run_metadata']['hardware']['gpu_name'],
    'compute_capability': evidence['run_metadata']['hardware']['compute_capability'],
    'precision': evidence['run_metadata']['hardware']['precision'],
    'trainable_percentage': evidence['run_metadata']['parameters']['trainable_percentage'],
    'training_loss': evidence['run_metadata']['training']['training_loss'],
    'adapter_size_mb': evidence['run_metadata']['adapter']['size_bytes'] / (1024**2),
    'telemetry_points': evidence['verification']['telemetry_points'],
    'same_prompts': evidence['verification']['same_prompts'],
}
print(json.dumps(summary, indent=2))


In [ ]:
from google.colab import files
files.download('artifacts/experiment-001/experiment_001_evidence.zip')
